# V02 — IAM e acesso seguro

## Objetivo da atividade

O participante identifica o menor acesso necessário, sabe onde solicitar permissão e não expõe segredos no código ou na gravação.

## Resultado esperado

Transformar tarefa em capacidade CDF mínima e registrar uma solicitação sanitizada.

## Ambiente

Cognite Data Fusion em **az-east1 / radix-dev**. O padrão é **DRY_RUN**; defina `CDF_EXECUTE=true` somente com credenciais aprovadas. Escrita exige também `CDF_ALLOW_WRITE=true`.

## Divisões da execução

1. **0:00–1:00** — Definir o princípio do menor privilégio e por que ambientes de dev, teste e produção são separados.
2. **1:00–3:00** — Mostrar a matriz: leitura de dados, escrita de dados, execução de transformação e administração. Explicar que acesso é concedido por necessidade de tarefa.
3. **3:00–5:00** — Demonstrar o conteúdo de uma solicitação correta: ambiente, grupo, justificativa, validade e aprovador.
4. **5:00–7:00** — Mostrar práticas proibidas: token no notebook, segredo em commit, captura de tela com URL privada e compartilhamento de usuário.
5. **7:00–8:30** — Mostrar o fluxo de revogação e como registrar bloqueio no backlog sem colocar dados sensíveis.
6. **8:30–10:00** — Recapitular e orientar o preenchimento da solicitação de acesso.

In [ ]:
from pathlib import Path
import sys, json, re
import pandas as pd

def locate_common(start=Path.cwd()):
    for root in [start, *start.parents]:
        candidate = root / '00-COMUM-CDF'
        if candidate.is_dir(): return candidate
    raise FileNotFoundError('Pasta 00-COMUM-CDF não encontrada.')

COMMON_DIR = locate_common()
if str(COMMON_DIR) not in sys.path: sys.path.insert(0, str(COMMON_DIR))
from cdf_training import build_cdf_client, execution_banner, safe_json, settings_from_env
SETTINGS = settings_from_env()
CDF = build_cdf_client(SETTINGS)
print(safe_json(execution_banner(SETTINGS)))
DATA_FILE = Path.cwd() / 'matriz-acessos.xlsx'


## Etapa 1 — 0:00–1:00

Definir o princípio do menor privilégio e por que ambientes de dev, teste e produção são separados.

In [ ]:
matrix = pd.read_excel(DATA_FILE, sheet_name='Matriz')
ENVIRONMENT_POLICY = {'dev': {'write': 'only with CDF_ALLOW_WRITE=true'}, 'test': {'write': 'approved test data only'}, 'prod': {'write': 'not part of onboarding'}}
print({'cluster': SETTINGS.cluster, 'project': SETTINGS.project, 'environment_policy': ENVIRONMENT_POLICY})


## Etapa 2 — 1:00–3:00

Mostrar a matriz: leitura de dados, escrita de dados, execução de transformação e administração. Explicar que acesso é concedido por necessidade de tarefa.

In [ ]:
TASK_TO_CDF_ACL = {'Consultar instâncias DMS': {'dataModelInstancesAcl': ['READ']}, 'Ler RAW': {'rawAcl': ['READ']}, 'Executar Transformation': {'transformationsAcl': ['READ', 'EXECUTE']}}
matrix['computed_capability'] = matrix['Tarefa'].map(lambda task: json.dumps(TASK_TO_CDF_ACL.get(task, {}), ensure_ascii=False))
display(matrix[['Tarefa', 'Capacidade CDF', 'Ação', 'computed_capability']])


## Etapa 3 — 3:00–5:00

Demonstrar o conteúdo de uma solicitação correta: ambiente, grupo, justificativa, validade e aprovador.

In [ ]:
def build_access_request(task, environment, justification, approver):
    capabilities = TASK_TO_CDF_ACL[task]
    return {'cluster': SETTINGS.cluster, 'project': SETTINGS.project, 'environment': environment, 'task': task, 'capabilities': capabilities, 'justification': justification, 'approver': approver, 'validity': 'training window'}
request = build_access_request('Consultar instâncias DMS', 'dev', 'Executar V02 com consulta limitada.', 'Tech Lead')
print(safe_json(request))


## Etapa 4 — 5:00–7:00

Mostrar práticas proibidas: token no notebook, segredo em commit, captura de tela com URL privada e compartilhamento de usuário.

In [ ]:
FORBIDDEN_MARKERS = [r'(?i)client_secret\s*=', r'(?i)authorization:\s*bearer', r'(?i)token\s*=', r'https://[^\s]+cognitedata\.com']
def scan_for_sensitive_content(text):
    return [pattern for pattern in FORBIDDEN_MARKERS if re.search(pattern, text)]
assert not scan_for_sensitive_content(safe_json(request))
print('Solicitação sanitizada: não contém token, segredo ou URL privada.')


## Etapa 5 — 7:00–8:30

Mostrar o fluxo de revogação e como registrar bloqueio no backlog sem colocar dados sensíveis.

In [ ]:
def build_revocation_record(request, reason):
    return {'project': request['project'], 'environment': request['environment'], 'capabilities_to_revoke': request['capabilities'], 'reason': reason, 'ticket_reference': '<link sanitizado do ticket>'}
revocation = build_revocation_record(request, 'Fim da janela de treinamento')
print(safe_json(revocation))


## Etapa 6 — 8:30–10:00

Recapitular e orientar o preenchimento da solicitação de acesso.

In [ ]:
if CDF is None:
    print('DRY_RUN: a revisão de acesso usaria CDF.iam.token.inspect().')
else:
    inspection = CDF.iam.token.inspect()
    print({'token_inspected': inspection is not None, 'evidence': 'registrar somente o resultado da revisão de capacidades'})


## Evidência para o backlog

Registrar no backlog a solicitação de acesso com ambiente, grupo mínimo, justificativa e link do ticket. Se já possuir acesso, anexar apenas a confirmação não sensível.

Não anexar token, segredo, URL privada, grupo real, log restrito ou dado produtivo.